In [ ]:
import pandas as pd

In [ ]:
df = pd.read_csv(
    r"C:\Users\lucij\Desktop\Leiden\Year 2\Thesis Project\2024_data\combined_dataset.csv"
)

In [ ]:
# separate df by instructions
df_inst0 = df[df['instructions'] == 0]
df_inst1 = df[df['instructions'] == 1]

In [ ]:
df_rt = df[
    (df['timeout'] == 0) &
    (df['response_time'].notna())
]

#### Checking timeouts

In [ ]:
print("Number of rows in df:", df.shape[0])
print("Number of rows in df_rt:", df_rt.shape[0])

In [ ]:
print(df_rt.shape[0]/df.shape[0])

print(df.shape[0] - df_rt.shape[0])

In [ ]:
#per instructions

df0_rt = df_inst0[
    (df_inst0['timeout'] == 0) &
    (df_inst0['response_time'].notna())
]

df1_rt = df_inst1[
    (df_inst1['timeout'] == 0) &
    (df_inst1['response_time'].notna())
]


In [ ]:
print(df0_rt.shape[0]/df_inst0.shape[0])

print(df_inst0.shape[0] - df0_rt.shape[0])

print(df_inst0.shape[0])
print(df0_rt.shape[0])

In [ ]:
print(df1_rt.shape[0]/df_inst1.shape[0])

print(df_inst1.shape[0] - df1_rt.shape[0])

print(df_inst1.shape[0])
print(df1_rt.shape[0])

In [ ]:
timeout_by_condition = df.groupby(['subject', 'instructions']).agg(
    total_trials=('timeout', 'count'),
    total_timeouts=('timeout', 'sum'),
).reset_index()

timeout_by_condition['timeout_rate'] = timeout_by_condition['total_timeouts'] / timeout_by_condition['total_trials']

print(timeout_by_condition.sort_values(['subject', 'instructions']))
pd.set_option('display.max_rows', None)
print(timeout_by_condition.sort_values('timeout_rate', ascending=False))

from scipy.stats import shapiro
group0 = timeout_by_condition[timeout_by_condition['instructions'] == 0]['timeout_rate']
group1 = timeout_by_condition[timeout_by_condition['instructions'] == 1]['timeout_rate']
print("Group 0:", shapiro(group0))
print("Group 1:", shapiro(group1))

In [ ]:
from scipy.stats import ttest_ind

t_stat, p = ttest_ind(group0, group1, equal_var=False)
print(f"Welch t-test: t = {t_stat:.3f}, p = {p:.3f}")

### RT

In [ ]:
import matplotlib.pyplot as plt
import matplotlib as mpl
import matplotlib.font_manager as fm
import seaborn as sns
import numpy as np

sns.set_style("ticks")
mpl.rcParams['font.family'] = 'Helvetica'
mpl.rcParams['font.sans-serif'] = ['Helvetica']

hv       = fm.FontProperties(family='Helvetica', size=16)
hv_large = fm.FontProperties(family='Helvetica', size=16)

rt_inst0 = df_inst0['response_time'].dropna()
rt_inst1 = df_inst1['response_time'].dropna()

bin_width = 0.7
data_min  = min(rt_inst0.min(), rt_inst1.min())
data_max  = max(rt_inst0.max(), rt_inst1.max())
bins      = np.arange(data_min, data_max + bin_width, bin_width)
bar_width = bin_width * 0.45

fig, ax = plt.subplots(figsize=(5.5, 5.5))

for rt, color, offset_sign in [
    (rt_inst0, "#daa800", -1),
    (rt_inst1, "#840000",  1),
]:
    counts, edges = np.histogram(rt, bins=bins)
    counts  = counts / len(rt) * 100
    centers = (edges[:-1] + edges[1:]) / 2
    ax.bar(
        centers + offset_sign * (bar_width / 2),
        counts,
        width=bar_width,
        color=color,
        alpha=1,
        edgecolor='white',
        linewidth=0.04,
    )

ax.set_xlabel('Response time (s)', color='black', labelpad=14,
              fontproperties=hv_large)
ax.set_ylabel('Frequency (%)', color='black', labelpad=14,
              fontproperties=hv_large)

ax.set_xlim(data_min - bin_width, data_max + bin_width)
ax.set_xticks(np.arange(0, 11, 2))
ax.set_xticklabels([f'{x:.0f}' for x in np.arange(0, 11, 2)])

ax.set_yticks([0, 20, 40])
ax.set_yticklabels(['0', '20', '40'])

for label in ax.get_xticklabels():
    label.set_fontproperties(hv)
for label in ax.get_yticklabels():
    label.set_fontproperties(hv)

ax.tick_params(axis='both', which='both', length=4, width=1,
               direction='out', pad=8, colors='black', labelsize = 14)

sns.despine(trim=False)
ax.spines['bottom'].set_position(('outward', 8))
ax.spines['left'].set_position(('outward', 8))
ax.spines['bottom'].set_bounds(0, 10)
plt.tight_layout()
plt.savefig('rt_histogram.png', dpi=600, bbox_inches='tight')
plt.show()

In [ ]:
overall_mean = df_rt['response_time'].mean()
overall_sd   = df_rt['response_time'].std()
print(f"Overall mean RT: {overall_mean:.3f} s (SD = {overall_sd:.3f} s)\n")

for cond, label in [(0, 'Min'), (1, 'Full')]:
    rt = df_rt[df_rt['instructions'] == cond]['response_time']
    n_below = (rt <= overall_mean).sum()
    pct_below = (rt <= overall_mean).mean() * 100
    print(f"{label} (N = {len(rt)} trials):")
    print(f"  {n_below} trials ({pct_below:.1f}%) at or below overall mean ({overall_mean:.3f} s)")

In [ ]:
print("Overall:")
print(f"  Mean RT:   {df_rt['response_time'].mean():.3f} s")
print(f"  Median RT: {df_rt['response_time'].median():.3f} s")

for cond, label in [(0, 'Min'), (1, 'Full')]:
    rt = df_rt[df_rt['instructions'] == cond]['response_time']
    mean_rt  = rt.mean()
    median_rt = rt.median()
    
    pct_at_mean   = (rt <= mean_rt).mean() * 100
    pct_at_median = (rt <= median_rt).mean() * 100
    n_at_mean     = (rt <= mean_rt).sum()
    n_at_median   = (rt <= median_rt).sum()
    
    print(f"\n{label} (N trials = {len(rt)}):")
    print(f"  Mean RT:   {mean_rt:.3f} s  → {n_at_mean} trials ({pct_at_mean:.1f}%) at or faster")
    print(f"  Median RT: {median_rt:.3f} s  → {n_at_median} trials ({pct_at_median:.1f}%) at or faster")
    print(f"  SD:        {rt.std():.3f} s")

### Contrasts / group

In [ ]:
# How many unique signed contrast values
num_unique = df_rt['stimContrast'].nunique()
print(f"Number of unique signed contrast values: {num_unique}")

# Optionally, list the unique values
unique_values = df_rt['stimContrast'].unique()
print(np.sort(unique_values))


In [ ]:
from statsmodels.stats.anova import AnovaRM

def test_rt_by_contrast(df_rt, label):
    df_rt = df_rt.copy()
    df_rt['signed_contrast'] = df_rt['signed_contrast'].round(4)
    
    participant_rt = (
        df_rt.groupby(['subject', 'signed_contrast'])['response_time']
             .mean()
             .reset_index()
    )
    
    # drop subjects missing any contrast level
    participant_rt = participant_rt.groupby('subject').filter(
        lambda x: len(x) == participant_rt['signed_contrast'].nunique()
    )
    
    aov = AnovaRM(participant_rt, depvar='response_time',
                  subject='subject', within=['signed_contrast'])
    print(f"\n{'═'*50}")
    print(f"Repeated measures ANOVA — {label}")
    print(f"{'═'*50}")
    print(aov.fit().summary())

test_rt_by_contrast(df0_rt, "Min (condition 0)")
test_rt_by_contrast(df1_rt, "Full (condition 1)")

### Accuracy

In [ ]:
from scipy.stats import ttest_ind

acc0_subj = df_rt[df_rt['instructions'] == 0].groupby('subject')['feedbackType_binary'].mean()
acc1_subj = df_rt[df_rt['instructions'] == 1].groupby('subject')['feedbackType_binary'].mean()

t_stat, p_val = ttest_ind(acc0_subj, acc1_subj, equal_var=False)

n0, n1 = len(acc0_subj), len(acc1_subj)
s0, s1 = acc0_subj.var(ddof=1), acc1_subj.var(ddof=1)
df_welch = (s0/n0 + s1/n1)**2 / ((s0/n0)**2/(n0-1) + (s1/n1)**2/(n1-1))

print(f"\nAccuracy comparison (Min vs Full):")
print(f"  Min:  M = {acc0_subj.mean():.3f}, SD = {acc0_subj.std():.3f}")
print(f"  Full: M = {acc1_subj.mean():.3f}, SD = {acc1_subj.std():.3f}")
print(f"  Welch's t({df_welch:.1f}) = {t_stat:.3f}, p = {p_val:.4e}")

### Outliers?

In [ ]:
participant_summary = df1_rt.groupby('subject').agg(
    mean_RT=('response_time', 'mean'),
    sd_RT=('response_time', 'std'),
    mean_acc=('feedbackType', 'mean')
).reset_index()

In [ ]:
# participants with very fast or very slow mean RTs
fast_threshold = participant_summary['mean_RT'].mean() - 3*participant_summary['mean_RT'].std()
slow_threshold = participant_summary['mean_RT'].mean() + 3*participant_summary['mean_RT'].std()
extreme_RT = participant_summary[(participant_summary['mean_RT'] < fast_threshold) |
                                 (participant_summary['mean_RT'] > slow_threshold)]

# participants with near-ceiling or near-floor accuracy
extreme_acc = participant_summary[(participant_summary['mean_acc'] < 0.5) | 
                                  (participant_summary['mean_acc'] > 0.95)]

print("Extreme RT participants:\n", extreme_RT)
print("Extreme accuracy participants:\n", extreme_acc)


In [ ]:
print("Min feedbackType values:", df0_rt['feedbackType'].unique())
print("Full feedbackType values:", df1_rt['feedbackType'].unique())

df0_rt = df0_rt.copy()
df1_rt = df1_rt.copy()
df0_rt['feedbackType_binary'] = (df0_rt['feedbackType'] == 1).astype(int)
df1_rt['feedbackType_binary'] = (df1_rt['feedbackType'] == 1).astype(int)

participant_acc0 = (
    df0_rt
    .groupby(['subject', 'signed_contrast'])['feedbackType_binary']
    .mean()
    .reset_index()
)

print("\nMin — accuracy at ±0.2 contrast:")
print(
    participant_acc0
    .query("signed_contrast == 0.2 or signed_contrast == -0.2")['feedbackType_binary']
    .describe()
)

participant_acc1 = (
    df1_rt
    .groupby(['subject', 'signed_contrast'])['feedbackType_binary']
    .mean()
    .reset_index()
)

print("\nFull — accuracy at ±0.2 contrast:")
print(
    participant_acc1
    .query("signed_contrast == 0.2 or signed_contrast == -0.2")['feedbackType_binary']
    .describe()
)

In [ ]:
from scipy.stats import ttest_ind

acc0_subj = participant_acc0.query("signed_contrast == 0.2 or signed_contrast == -0.2").groupby('subject')['feedbackType_binary'].mean()
acc1_subj = participant_acc1.query("signed_contrast == 0.2 or signed_contrast == -0.2").groupby('subject')['feedbackType_binary'].mean()

stat, p = ttest_ind(acc0_subj, acc1_subj, equal_var=False)  # equal_var=False = Welch
print(f"Welch t-test: t = {stat:.3f}, p = {p:.4e}")